# 01 · Data cleaning and quality control

This notebook audits the public SWELL-KW feature workbook used in the case study. It does not claim ownership of data collection.

**Source:** SWELL Knowledge Work Dataset, DOI [10.17026/DANS-X55-69ZP](https://doi.org/10.17026/DANS-X55-69ZP)  
**Input:** `Behavioral-features - per minute.xlsx`, sheet `SWELLdata`

The questionnaire fields contain one post-condition value repeated on every minute. The block-level table below therefore keeps one questionnaire response and averages objective features within each participant-condition block.

In [1]:
from pathlib import Path
import os, sys, json
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
from data_cleaning import load_and_clean, validate_structure, participant_condition_table, sha256

raw_path = Path(os.environ.get("SWELL_DATA_PATH", ROOT / "data" / "raw" / "Behavioral-features - per minute.xlsx"))
if not raw_path.exists():
    raise FileNotFoundError(
        "Download the public workbook from DOI 10.17026/DANS-X55-69ZP, place it in data/raw, "
        "or set SWELL_DATA_PATH."
    )
print(f"Input: {raw_path.name}")
print(f"SHA-256: {sha256(raw_path)}")

Input: Behavioral-features - per minute.xlsx
SHA-256: 9ae1fc82569e9b16f363fb8b0106c859422a6a379ac577b4e8268f51cf20a044


In [2]:
data = load_and_clean(raw_path)
qc = validate_structure(data)
qc["source_sha256"] = sha256(raw_path)
work = data[data["Condition"].isin(["N", "I", "T"])].copy()
blocks = participant_condition_table(work)

pd.DataFrame({
    "audit_item": ["all rows", "work rows", "relaxation rows", "participants", "participant-condition blocks"],
    "value": [len(data), len(work), (data.Condition == "R").sum(), work.PP.nunique(), len(blocks)],
})

Out[0]: 
                     audit_item  value
0                      all rows   3139
1                     work rows   2688
2               relaxation rows    451
3                  participants     25
4  participant-condition blocks     75


## Experimental order audit

In [3]:
order = (blocks.sort_values(["PP", "Blok"]).groupby("PP")["Condition"].agg("".join).value_counts())
print(order.to_string())
print(f"Neutral always block 1: {(blocks.loc[blocks.Condition == 'N', 'Blok'] == 1).all()}")

Condition
NIT    13
NTI    12
Neutral always block 1: True


Neutral was always first; only interruptions and time pressure were counterbalanced. This prevents a clean separation of Neutral-versus-stressor effects from order, practice, fatigue, or sensor drift.

## Missingness and repeated subjective labels

In [4]:
from config import BEHAVIOR_FEATURES, PHYSIOLOGY_FEATURES, SUBJECTIVE_FEATURES

missing = (work[BEHAVIOR_FEATURES + PHYSIOLOGY_FEATURES + SUBJECTIVE_FEATURES]
           .isna().mean().mul(100).sort_values(ascending=False)
           .rename("missing_percent").to_frame())
missing.head(12).round(1)

Out[0]: 
                 missing_percent
RMSSD                       52.9
HR                          52.9
SCL                         19.9
NasaTLX                      3.8
SnSpecialKeys                1.2
SnDirectionKeys              1.2
SnErrorKeys                  1.2
SnShortcutKeys               1.2
SnMouseAct                   1.2
SnLeftClicked                1.2
SnKeyStrokes                 1.2
SnChars                      1.2


In [5]:
unique_within_block = work.groupby(["PP", "Condition"])[SUBJECTIVE_FEATURES].nunique(dropna=True)
print(f"Maximum unique questionnaire values within a participant-condition block: {int(unique_within_block.max().max())}")
print("One value confirms that minute-level rows must not be treated as independent questionnaire observations.")

Maximum unique questionnaire values within a participant-condition block: 1
One value confirms that minute-level rows must not be treated as independent questionnaire observations.


## Write analytic files

In [6]:
from data_cleaning import run
run(raw_path, ROOT / "data" / "processed")
print("Wrote minute_work.csv, participant_condition.csv, and qc_summary.json")

Wrote minute_work.csv, participant_condition.csv, and qc_summary.json
